Imports and dates

In [1]:
import sys
from pathlib import Path

# Add repo root to PYTHONPATH
HERE = Path().resolve()
PROJECT_ROOT = HERE.parents[0]   # notebooks → repo root
sys.path.insert(0, str(PROJECT_ROOT))


from prm_opt.run_s25 import run_s25_s1, run_s25_s2
from prm_opt.run_s26 import run_s26_s1, run_s26_s2
from prm_opt.config import PlanningToggles

# -----------------------------
# DATE RANGES
# -----------------------------
START_S25 = "2025-03-30"
END_S25   = "2025-04-02" #"2025-10-26"

START_S26 = "2025-03-29"
END_S26   = "2025-10-24"

Common toggles


In [2]:

toggles = PlanningToggles(
    sla_buffer_mins=5,     # tighten SLA by 5 mins
    handover_mins=10,      # 10-min handover overlap for Amb+Mini
    max_late_mins=360
)

Run S25 Scenario 1 (baseline)

In [3]:
out_s25_s1 = run_s25_s1(START_S25, END_S25, toggles=toggles)

out_s25_s1["summary"]



{'PeakAmb': 5,
 'PeakMini': 1,
 'PeakDrivers': 5,
 'PeakAmb_bucket': Timestamp('2025-03-30 11:00:00'),
 'PeakMini_bucket': Timestamp('2025-03-30 09:15:00'),
 'PeakDrivers_bucket': Timestamp('2025-03-30 11:00:00'),
 'CurrentAmb': 14,
 'GapAmb': -9,
 'CurrentMini': 3,
 'GapMini': -2}

In [4]:
out_s25_s1["ambulift_curve"]


s
2025-03-30 00:00:00    1
2025-03-30 00:15:00    3
2025-03-30 00:30:00    2
2025-03-30 00:45:00    2
2025-03-30 04:15:00    3
                      ..
2025-04-01 22:30:00    1
2025-04-01 22:45:00    1
2025-04-01 23:00:00    1
2025-04-01 23:15:00    1
2025-04-02 00:00:00    0
Name: _amb, Length: 241, dtype: int64

In [5]:
out_s25_s1["driver_curve"].head()

s
2025-03-30 00:00:00    1
2025-03-30 00:15:00    3
2025-03-30 00:30:00    2
2025-03-30 00:45:00    2
2025-03-30 04:15:00    3
dtype: int64

In [6]:

from prm_opt.ingest_s25 import ingest_s25
from prm_opt.build_jobs import build_jobs

df = ingest_s25("2025-06-10", "2025-06-12")
jobs = build_jobs(df, bucket="15min", toggles=toggles)

jobs[jobs["t"].isna()].head(20)



,Passenger ID,flight_key,release_time,t,s,dir,zone,Stand,class,Airline Code,...,base_duration_mins,is_spin,SSR numeric,IsEffectiveRemote,IsArrival,Turnaround PRM Count,Concurrent Stress,PRM Flight Count,Has Own Chair,IsAdhoc
j,,,,,,,,,,,,,,,,,,,,,


In [7]:
print(len(jobs[jobs["t"].isna()]))
len(jobs[jobs["t"].notna()])

0


1226

Run S25 Scenario 2 (optimised)

In [8]:
out_s25_s2 = run_s25_s2(
    start=START_S25,
    end=END_S25,
    solver_name="highs",
    toggles=toggles,
)

out_s25_s2["summary"]

spin_removed > total amb minutes: 0
[]
ERROR: Rule failed when generating expression for Constraint
'FutureVehicleConstraint_Amb_2025-03-29 23:45:00' with index None: KeyError:
"Index '(None, 'Amb_C0', 'A3_632_2025-03-31 09:00:00', Timestamp('2025-03-29
23:45:00'))' is not valid for indexed component 'k'"
ERROR: Constructing component ''FutureVehicleConstraint_Amb_2025-03-29
23:45:00'' from data=None failed:
        KeyError: "Index '(None, 'Amb_C0', 'A3_632_2025-03-31 09:00:00',
        Timestamp('2025-03-29 23:45:00'))' is not valid for indexed component
        'k'"


KeyError: "Index '(None, 'Amb_C0', 'A3_632_2025-03-31 09:00:00', Timestamp('2025-03-29 23:45:00'))' is not valid for indexed component 'k'"

Inspect raw optimisation decisions

In [ ]:
df_jobs_25 = out_s25_s2["job_assignments"]
df_jobs_25.head()


In [ ]:

# Vertical jobs that used Mini horizontally
out_s25_s2["sanity_checks"]["vertical_with_mini"]



In [ ]:
# SLA breaches by direction
out_s25_s2["sanity_checks"]["sla_breaches_by_dir"]

vehicle allocations

In [ ]:
df_veh_25 = out_s25_s2["vehicle_allocations"]
df_veh_25.head()


Run S26 Scenario 1

In [ ]:

out_s26_s1 = run_s26_s1(
    start=START_S26,
    end=END_S26,
    penetration_rates=penetration_rates,
    ssr_mix=ssr_mix,
    stand_actuals=stand_actuals,
    stand_dist=stand_dist,
    service_time_params=service_time_params,
    chocks_offset_params=chocks_offset_params,
    toggles=toggles,
)

out_s26_s1["summary"]
out_s26_s1["ambulift_curve"].head()
out_s26_s1["driver_curve"].head()


Run S26 scenario 2

In [ ]:

out_s26_s2 = run_s26_s2(
    start=START_S26,
    end=END_S26,
    penetration_rates=penetration_rates,
    ssr_mix=ssr_mix,
    stand_actuals=stand_actuals,
    stand_dist=stand_dist,
    service_time_params=service_time_params,
    chocks_offset_params=chocks_offset_params,
    solver_name="highs",
    toggles=toggles,
)

out_s26_s2["summary"]
